<!-- ---
title: Cut LLM time-to-first-token on Amazon SageMaker with Hugging Face vLLM and prefix-aware routing
category: text-generation
navigation: advanced
--- -->

# Cut LLM time-to-first-token on Amazon SageMaker with Hugging Face vLLM and prefix-aware routing

A warm vLLM prefix cache only helps when later requests reach the replica that owns it. In this tutorial, we deploy [`Qwen/Qwen3-8B`](https://huggingface.co/Qwen/Qwen3-8B) on two Amazon SageMaker AI endpoint instances, enable vLLM automatic prefix caching, and compare SageMaker's `RANDOM`, `LEAST_OUTSTANDING_REQUESTS`, and `PREFIX_AWARE` routing strategies.

You will:

- Deploy the model with SageMaker Python SDK v3 `ModelBuilder` and the Hugging Face vLLM DLC.
- Replay an enterprise-style workload with a 1K–8K-token shared system block.
- Measure client-side TTFT p50/p90/p95, streamed-chunk latency, token throughput, and request throughput.
- Enable SageMaker detailed inference observability for native vLLM TTFT, ITL, KV-cache, queue-depth, batch-size, TPS, and GPU metrics.
- Sweep shared-prefix length and `ConcurrencyThreshold` to find where affinity pays off.
- Verify that routing leaves structured output, tool calls, and deterministic answers unchanged.

This experiment creates billable GPU endpoints with two instances. The notebook deploys one routing configuration at a time and deletes it before continuing, but you should still run the cleanup cell if execution is interrupted.

## Why prefix-aware routing changes TTFT

vLLM stores the keys and values produced while prefilling a prompt. With automatic prefix caching enabled, a replica can skip most of that prefill when a later request begins with the same token sequence. Random routing and least-outstanding-requests can send that request to another replica, where the same cache entry does not exist. Prefix-aware routing adds affinity: matching request prefixes prefer the same replica until its configured concurrency threshold is reached, at which point SageMaker temporarily favors spare capacity.

<!-- IMAGE PLACEHOLDER: add ./assets/prefix-routing.png to compare cache-local and cache-miss request paths. -->
<!-- ![Prefix-aware routing keeps shared prompts near their warm KV cache](./assets/prefix-routing.png) -->

Prefix-aware routing improves locality, not model computation itself. It is most useful when prompts have long, stable prefixes and the serving engine has prefix caching enabled. Short or mostly unique prompts have less reusable prefill work, while an overly high concurrency threshold can create a hotspot.

## Setup

Use an AWS Region that offers the selected GPU instance and the Hugging Face vLLM DLC. Your identity needs permissions for SageMaker models, endpoint configurations, endpoints, runtime invocation, ECR image pulls, and CloudWatch metric queries. In SageMaker Studio, the execution role can usually be discovered automatically.

In [ ]:
%pip install -q "sagemaker>=3.22.0" "boto3>=1.43.86" "transformers>=5.0" pandas matplotlib seaborn requests

In [ ]:
import json
import os
from time import strftime

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

REGION = boto3.Session().region_name or os.environ.get("AWS_REGION", "us-east-1")
boto_sess = boto3.Session(region_name=REGION)
session = Session(boto_session=boto_sess)
sm = boto_sess.client("sagemaker")
runtime = boto_sess.client("sagemaker-runtime")

try:
    role = get_execution_role(sagemaker_session=session)
except Exception:
    role_name = os.environ.get("SAGEMAKER_EXECUTION_ROLE_NAME", "sagemaker_execution_role")
    role = boto_sess.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]

print(f"Region: {REGION}")
print(f"Role: {role}")

## Build Qwen3-8B with prefix caching

We use the newest Hugging Face-specific vLLM DLC required by this tutorial, not the newer generic vLLM image. The image maps `SM_VLLM_*` variables to upstream vLLM server arguments, so `SM_VLLM_ENABLE_PREFIX_CACHING=true` enables `--enable-prefix-caching`.

`Qwen/Qwen3-8B` keeps the serving stack aligned with other current Hugging Face SageMaker examples. The two-instance endpoint is deliberately larger than needed for one 8B model: replica-local cache behavior only becomes visible when a router has a choice of destinations.

In [ ]:
from sagemaker.serve import ModelBuilder, ModelServer

MODEL_ID = "Qwen/Qwen3-8B"
INSTANCE_TYPE = "ml.g5.xlarge"
INSTANCE_COUNT = 2
HF_VLLM_IMAGE = (
    f"763104351884.dkr.ecr.{REGION}.amazonaws.com/"
    "huggingface-vllm:"
    "0.28.0-transformers5.15.0-gpu-py312-cu130-ubuntu24.04"
)

RUN_ID = strftime("%Y%m%d-%H%M%S")
MODEL_NAME = f"qwen3-prefix-routing-{RUN_ID}"

builder = ModelBuilder(
    model=MODEL_ID,
    image_uri=HF_VLLM_IMAGE,
    role_arn=role,
    sagemaker_session=session,
    instance_type=INSTANCE_TYPE,
    model_server=ModelServer.VLLM,
    env_vars={
        "SM_VLLM_HOST": "0.0.0.0",
        "SM_VLLM_ENABLE_PREFIX_CACHING": "true",
        "SM_VLLM_MAX_MODEL_LEN": "16384",
        "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.90",
        "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
        "SM_VLLM_TOOL_CALL_PARSER": "hermes",
    },
)

built_model = builder.build(model_name=MODEL_NAME)
print(f"Model resource: {built_model.model_name}")
print(f"Image: {HF_VLLM_IMAGE}")

## Create endpoint configurations with explicit routing

Routing is configured on a production variant. The generated SageMaker SDK v3 resource shape may lag this new API, so this notebook intentionally uses the SageMaker client only for `CreateEndpointConfig`; model creation remains on SDK v3 `ModelBuilder`.

For `InvokeEndpoint` and `InvokeEndpointWithResponseStream`, `PrefixLength` is measured in request-body **bytes**. For SageMaker's OpenAI-compatible API, it is measured in **characters from message text**. This notebook uses `InvokeEndpointWithResponseStream`, even though its JSON body follows the OpenAI schema, so the byte rule applies. Valid values are 1,024–65,536.

`ConcurrencyThreshold` is the maximum in-flight request count on the affinity-selected replica before SageMaker temporarily routes to a replica with spare capacity. We keep `PrefixLength=8192` bytes and sweep the threshold. Detailed observability is enabled at a 10-second interval so the benchmark can pair client timings with native vLLM and GPU metrics.

In [ ]:
ROUTING_PREFIX_BYTES = 8192
METRIC_PUBLISH_FREQUENCY = 10


def create_benchmark_endpoint(strategy: str, concurrency_threshold: int | None = None):
    suffix = strategy.lower().replace("_", "-")
    if concurrency_threshold is not None:
        suffix += f"-ct{concurrency_threshold}"
    config_name = f"prefix-routing-config-{suffix}-{RUN_ID}"
    endpoint_name = f"prefix-routing-{suffix}-{RUN_ID}"

    routing_config = {"RoutingStrategy": strategy}
    if strategy == "PREFIX_AWARE":
        routing_config["PrefixAwareRoutingConfig"] = {
            "PrefixLength": ROUTING_PREFIX_BYTES,
            "ConcurrencyThreshold": concurrency_threshold,
        }

    sm.create_endpoint_config(
        EndpointConfigName=config_name,
        ProductionVariants=[{
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": INSTANCE_COUNT,
            "InstanceType": INSTANCE_TYPE,
            "ContainerStartupHealthCheckTimeoutInSeconds": 900,
            "InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1",
            "RoutingConfig": routing_config,
        }],
        MetricsConfig={
            "EnableDetailedObservability": True,
            "EnableEnhancedMetrics": True,
            "MetricPublishFrequencyInSeconds": METRIC_PUBLISH_FREQUENCY,
        },
    )
    sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
    sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
    return endpoint_name, config_name


def delete_benchmark_endpoint(endpoint_name: str, config_name: str):
    sm.delete_endpoint(EndpointName=endpoint_name)
    sm.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)
    sm.delete_endpoint_config(EndpointConfigName=config_name)

## Build a repeatable enterprise workload

Each request starts with a shared system block containing support policy, tool guidance, schemas, and operating rules, followed by a short unique user request. We generate variants near 1K, 4K, and 8K tokens. The short case is important: with an 8,192-byte routing prefix, unique user text can enter the routing key when the shared system block is too short, reducing affinity. The long cases keep the routing key inside shared content and make skipped prefill work valuable.

<!-- IMAGE PLACEHOLDER: add ./assets/workload-prefixes.png to show shared and unique request regions. -->
<!-- ![Shared-prefix benchmark request layout](./assets/workload-prefixes.png) -->

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

POLICY_BLOCK = """
Customer Support Operating Policy
- Protect customer data and never reveal credentials, private identifiers, or internal notes.
- Confirm the account and product before recommending account-specific actions.
- Treat refunds, credits, and replacements as separate workflows with explicit eligibility checks.
- Escalate safety, fraud, legal, and repeated-failure cases to a human specialist.
- Use only documented tools. Validate tool arguments against their JSON schemas.
- Return concise steps, cite the policy section used, and state any unresolved assumptions.
Tool schema: lookup_order(order_id: string), lookup_policy(topic: string), create_case(summary: string, priority: enum[low, normal, high]).
Response schema: {summary: string, actions: string[], escalation_required: boolean, policy_basis: string[]}.
""".strip()


def make_shared_prefix(target_tokens: int) -> tuple[str, int]:
    header = f"Policy bundle for {target_tokens}-token benchmark cohort.\n"
    text = header
    while len(tokenizer.encode(text, add_special_tokens=False)) < target_tokens:
        text += "\n\n" + POLICY_BLOCK
    token_ids = tokenizer.encode(text, add_special_tokens=False)[:target_tokens]
    prefix = tokenizer.decode(token_ids, skip_special_tokens=True)
    return prefix, len(tokenizer.encode(prefix, add_special_tokens=False))


PREFIX_TARGETS = [1024, 4096, 8192]
SHARED_PREFIXES = {
    target: make_shared_prefix(target) for target in PREFIX_TARGETS
}
USER_REQUESTS = [
    "Order A1001 arrived damaged. Summarize the next steps.",
    "Order A1002 is delayed by five days. What should the agent do?",
    "A customer cannot sign in after a password reset. Provide safe troubleshooting steps.",
    "A customer disputes a renewal charge. Identify the correct workflow.",
    "A device overheated during charging. Decide whether to escalate.",
    "A customer asks an agent to reveal another user's shipping address. Respond appropriately.",
    "Order A1007 contains the wrong item. Draft a concise resolution plan.",
    "A customer reports repeated payment failures. Recommend the next action.",
]

for target, (prefix, actual) in SHARED_PREFIXES.items():
    print(target, "target tokens ->", actual, "actual tokens,", len(prefix.encode("utf-8")), "bytes")

## Measure streamed responses

`InvokeEndpointWithResponseStream` lets us timestamp the first generated chunk instead of treating full-response latency as TTFT. Client-side streamed-chunk intervals are useful for comparing runs, but they are not exact token-level ITL because one SSE chunk can contain more than one token. The detailed observability section later queries vLLM's native ITL histogram.

In [ ]:
from time import perf_counter


def stream_chat(endpoint_name: str, system_prompt: str, user_prompt: str, request_id: int) -> dict:
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Request {request_id}: {user_prompt}"},
        ],
        "temperature": 0.0,
        "max_tokens": 64,
        "stream": True,
        "stream_options": {"include_usage": True},
        "chat_template_kwargs": {"enable_thinking": False},
    }
    body = json.dumps(payload, separators=(",", ":")).encode("utf-8")
    started = perf_counter()
    response = runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=body,
        CustomAttributes="route=/v1/chat/completions",
    )

    buffer = ""
    first_chunk_at = None
    chunk_times = []
    output_parts = []
    usage = {}

    for event in response["Body"]:
        if "PayloadPart" not in event:
            raise RuntimeError(f"Streaming invocation failed: {event}")
        buffer += event["PayloadPart"]["Bytes"].decode("utf-8")
        while "\n\n" in buffer:
            block, buffer = buffer.split("\n\n", 1)
            for line in block.splitlines():
                if not line.startswith("data:"):
                    continue
                data = line.removeprefix("data:").strip()
                if not data or data == "[DONE]":
                    continue
                chunk = json.loads(data)
                if chunk.get("usage"):
                    usage = chunk["usage"]
                choices = chunk.get("choices") or []
                if not choices:
                    continue
                delta = choices[0].get("delta", {})
                piece = delta.get("content") or delta.get("reasoning_content") or ""
                if piece:
                    now = perf_counter()
                    first_chunk_at = first_chunk_at or now
                    chunk_times.append(now)
                    output_parts.append(piece)

    finished = perf_counter()
    if first_chunk_at is None:
        raise RuntimeError("The stream completed without a generated text chunk.")
    chunk_intervals = [
        later - earlier for earlier, later in zip(chunk_times, chunk_times[1:])
    ]
    duration = finished - started
    prompt_tokens = usage.get("prompt_tokens", 0)
    completion_tokens = usage.get("completion_tokens", 0)
    return {
        "request_id": request_id,
        "ttft_ms": 1000 * (first_chunk_at - started),
        "e2e_ms": 1000 * duration,
        "stream_chunk_itl_ms": 1000 * sum(chunk_intervals) / len(chunk_intervals) if chunk_intervals else None,
        "input_tokens": prompt_tokens,
        "output_tokens": completion_tokens,
        "input_tokens_per_second": prompt_tokens / duration if duration else None,
        "output_tokens_per_second": completion_tokens / duration if duration else None,
        "output": "".join(output_parts),
    }

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

REQUESTS_PER_RUN = 48
CONCURRENCY = 16
WARMUP_REQUESTS = 12


def benchmark_workload(endpoint_name: str, target_tokens: int) -> tuple[pd.DataFrame, float]:
    system_prompt, actual_tokens = SHARED_PREFIXES[target_tokens]

    # Warm caches and the model server before measurements. Random routing gets
    # enough attempts to reach both replicas; prefix-aware routing builds affinity.
    for request_id in range(WARMUP_REQUESTS):
        stream_chat(
            endpoint_name,
            system_prompt,
            USER_REQUESTS[request_id % len(USER_REQUESTS)],
            -request_id - 1,
        )

    started = perf_counter()
    rows = []
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
        futures = [
            pool.submit(
                stream_chat,
                endpoint_name,
                system_prompt,
                USER_REQUESTS[request_id % len(USER_REQUESTS)],
                request_id,
            )
            for request_id in range(REQUESTS_PER_RUN)
        ]
        for future in as_completed(futures):
            rows.append(future.result())
    wall_seconds = perf_counter() - started

    frame = pd.DataFrame(rows)
    frame["target_prefix_tokens"] = target_tokens
    frame["actual_prefix_tokens"] = actual_tokens
    frame["request_throughput_rps"] = len(frame) / wall_seconds
    return frame, wall_seconds


def summarize_requests(frame: pd.DataFrame) -> dict:
    return {
        "requests": len(frame),
        "ttft_p50_ms": frame["ttft_ms"].quantile(0.50),
        "ttft_p90_ms": frame["ttft_ms"].quantile(0.90),
        "ttft_p95_ms": frame["ttft_ms"].quantile(0.95),
        "stream_chunk_itl_p50_ms": frame["stream_chunk_itl_ms"].median(),
        "input_tps_mean": frame["input_tokens_per_second"].mean(),
        "output_tps_mean": frame["output_tokens_per_second"].mean(),
        "request_throughput_rps": frame["request_throughput_rps"].iloc[0],
    }

## Check application behavior

Routing must not change what the application receives. For each endpoint configuration, the quality gate checks a constrained JSON response, a required tool call, and a temperature-zero answer. It validates the response shapes immediately and compares canonical outputs after all strategies finish. Exact agreement is expected here because sampling is disabled; with nonzero temperature, use a task-specific score and an acceptance interval instead of string equality.

In [ ]:
def invoke_chat(endpoint_name: str, payload: dict) -> dict:
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload),
        CustomAttributes="route=/v1/chat/completions",
    )
    return json.loads(response["Body"].read())


def run_quality_suite(endpoint_name: str) -> dict:
    base = {
        "model": MODEL_ID,
        "temperature": 0.0,
        "max_tokens": 128,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    schema = {
        "type": "object",
        "properties": {
            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
            "route": {"type": "string", "enum": ["self_service", "human"]},
        },
        "required": ["severity", "route"],
        "additionalProperties": False,
    }
    structured = invoke_chat(endpoint_name, {
        **base,
        "messages": [{"role": "user", "content": "Classify: a charging device emitted smoke."}],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "triage", "schema": schema},
        },
    })
    structured_text = structured["choices"][0]["message"]["content"]
    structured_value = json.loads(structured_text)
    structured_valid = (
        set(structured_value) == {"severity", "route"}
        and structured_value["severity"] in schema["properties"]["severity"]["enum"]
        and structured_value["route"] in schema["properties"]["route"]["enum"]
    )

    tool_response = invoke_chat(endpoint_name, {
        **base,
        "messages": [{"role": "user", "content": "Look up order A1001."}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "lookup_order",
                "description": "Look up an order by id.",
                "parameters": {
                    "type": "object",
                    "properties": {"order_id": {"type": "string"}},
                    "required": ["order_id"],
                },
            },
        }],
        "tool_choice": "required",
    })
    tool_calls = tool_response["choices"][0]["message"].get("tool_calls") or []
    tool_args = json.loads(tool_calls[0]["function"]["arguments"]) if tool_calls else {}
    tool_valid = bool(
        tool_calls
        and tool_calls[0]["function"]["name"] == "lookup_order"
        and tool_args.get("order_id") == "A1001"
    )

    answer_response = invoke_chat(endpoint_name, {
        **base,
        "messages": [{"role": "user", "content": "Reply with only the capital of France."}],
    })
    answer = answer_response["choices"][0]["message"]["content"].strip().lower()

    return {
        "structured_valid": structured_valid,
        "structured_output": json.dumps(structured_value, sort_keys=True),
        "tool_valid": tool_valid,
        "tool_output": json.dumps({"name": "lookup_order", "arguments": tool_args}, sort_keys=True),
        "task_success": answer.strip(". ") == "paris",
        "deterministic_answer": answer,
    }

## Run the routing and threshold sweep

The matrix keeps the model, image, two-instance hardware, prompts, generation parameters, warmup, request count, and concurrency fixed. It compares both non-affinity strategies with prefix-aware routing at concurrency thresholds 4, 8, and 16. Each routing configuration receives a fresh endpoint so caches do not leak across strategies; all three prefix lengths run on that endpoint before it is deleted.

Start with the single `PREFIX_AWARE, 8` configuration if you want a lower-cost smoke test. The full matrix creates five two-instance endpoints sequentially and can take significant time because every endpoint downloads and loads the model.

In [ ]:
from datetime import datetime, timezone
from time import sleep

EXPERIMENTS = [
    ("RANDOM", None),
    ("LEAST_OUTSTANDING_REQUESTS", None),
    ("PREFIX_AWARE", 4),
    ("PREFIX_AWARE", 8),
    ("PREFIX_AWARE", 16),
]

request_frames = []
summary_rows = []
quality_rows = []
run_windows = []

for strategy, threshold in EXPERIMENTS:
    endpoint_name, config_name = create_benchmark_endpoint(strategy, threshold)
    label = strategy if threshold is None else f"{strategy}_CT{threshold}"
    print(f"Running {label} on {endpoint_name}")
    try:
        for target_tokens in PREFIX_TARGETS:
            window_start = datetime.now(timezone.utc)
            frame, wall_seconds = benchmark_workload(endpoint_name, target_tokens)
            window_end = datetime.now(timezone.utc)

            frame["routing_strategy"] = strategy
            frame["concurrency_threshold"] = threshold
            frame["experiment"] = label
            request_frames.append(frame)

            summary_rows.append({
                "routing_strategy": strategy,
                "concurrency_threshold": threshold,
                "experiment": label,
                "target_prefix_tokens": target_tokens,
                **summarize_requests(frame),
            })
            run_windows.append({
                "endpoint_name": endpoint_name,
                "routing_strategy": strategy,
                "concurrency_threshold": threshold,
                "experiment": label,
                "target_prefix_tokens": target_tokens,
                "start": window_start,
                "end": window_end,
            })

        quality_rows.append({
            "routing_strategy": strategy,
            "concurrency_threshold": threshold,
            "experiment": label,
            **run_quality_suite(endpoint_name),
        })
        sleep(METRIC_PUBLISH_FREQUENCY + 5)
    finally:
        delete_benchmark_endpoint(endpoint_name, config_name)

requests_df = pd.concat(request_frames, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
quality_df = pd.DataFrame(quality_rows)
summary_df.sort_values(["target_prefix_tokens", "experiment"])

## Compare client-side latency and throughput

Read each prefix-length panel separately. Prefix-aware routing should have little advantage when the configured routing prefix extends into unique request content. As the shared prefix grows beyond the 8,192-byte routing key, cache locality should become more stable and skipped prefill should have a larger effect. The threshold sweep shows the competing cost: low thresholds protect load balance but spill affinity sooner, while high thresholds preserve cache locality but can queue work behind one replica.

<!-- PLOT PLACEHOLDER: replace or supplement the generated chart with ./assets/ttft-by-routing.png. -->
<!-- ![TTFT percentiles by routing strategy and shared-prefix length](./assets/ttft-by-routing.png) -->

<!-- PLOT PLACEHOLDER: replace or supplement the generated chart with ./assets/throughput-by-routing.png. -->
<!-- ![Request and token throughput by routing strategy](./assets/throughput-by-routing.png) -->

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

latency_long = summary_df.melt(
    id_vars=["experiment", "target_prefix_tokens"],
    value_vars=["ttft_p50_ms", "ttft_p90_ms", "ttft_p95_ms"],
    var_name="percentile",
    value_name="ttft_ms",
)
latency_long["percentile"] = latency_long["percentile"].str.extract(r"(p\d+)")

chart = sns.catplot(
    data=latency_long,
    x="experiment",
    y="ttft_ms",
    hue="percentile",
    col="target_prefix_tokens",
    kind="bar",
    sharey=False,
    height=4,
    aspect=1.15,
)
chart.set_axis_labels("Routing configuration", "TTFT (ms)")
chart.set_titles("Shared prefix: {col_name} tokens")
for axis in chart.axes.flat:
    axis.tick_params(axis="x", rotation=35)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(
    data=summary_df,
    x="target_prefix_tokens",
    y="request_throughput_rps",
    hue="experiment",
    marker="o",
    ax=axes[0],
)
sns.lineplot(
    data=summary_df,
    x="target_prefix_tokens",
    y="output_tps_mean",
    hue="experiment",
    marker="o",
    ax=axes[1],
)
axes[0].set(title="Request throughput", xlabel="Shared-prefix tokens", ylabel="Requests/s")
axes[1].set(title="Output-token throughput", xlabel="Shared-prefix tokens", ylabel="Tokens/s per request")
plt.tight_layout()
plt.show()

## Add native vLLM and GPU observability

Client timings tell us whether users improved; engine metrics explain why. SageMaker detailed observability scrapes vLLM and GPU telemetry through OpenTelemetry. The endpoint configuration above enables collection. To query those metrics with PromQL, your account must also have CloudWatch OTel enrichment enabled, and the notebook identity needs `cloudwatch:GetMetricData` and `cloudwatch:ListMetrics`.

You can inspect the managed **SageMaker AI Insights** dashboard in CloudWatch, or run the signed PromQL queries below. They collect native vLLM TTFT and ITL histograms, input/output token rates, KV-cache utilization, queue depth, running batch size, and GPU utilization over each client benchmark window. No custom metrics server is required.

The first metric publication can lag a short run. If a query returns no samples, widen the time padding, increase `REQUESTS_PER_RUN`, or inspect the Insights dashboard after a few minutes.

In [ ]:
import math
from datetime import timedelta
from urllib.parse import urlencode

import requests as http_requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest


def query_promql_range(query: str, start: datetime, end: datetime, step: int = 10) -> list:
    url = f"https://monitoring.{REGION}.amazonaws.com/api/v1/query_range"
    form = urlencode({
        "query": query,
        "start": start.timestamp(),
        "end": end.timestamp(),
        "step": step,
    })
    credentials = boto_sess.get_credentials().get_frozen_credentials()
    aws_request = AWSRequest(
        method="POST",
        url=url,
        data=form.encode("utf-8"),
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        },
    )
    SigV4Auth(credentials, "monitoring", REGION).add_auth(aws_request)
    response = http_requests.post(
        url,
        data=form.encode("utf-8"),
        headers=dict(aws_request.headers),
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("status") != "success":
        raise RuntimeError(payload)
    return payload["data"].get("result", [])


def metric_values(series: list) -> list[float]:
    values = []
    for item in series:
        for _, value in item.get("values", []):
            number = float(value)
            if math.isfinite(number):
                values.append(number)
    return values


def promql_queries(endpoint_name: str) -> dict[str, str]:
    selector = f"'aws.sagemaker.endpoint.name'=\"{endpoint_name}\""
    return {
        "native_ttft_p50_ms": f"1000 * histogram_quantile(0.50, sum by (le) (rate(vllm_time_to_first_token_seconds_bucket{{{selector}}}[1m])))",
        "native_ttft_p90_ms": f"1000 * histogram_quantile(0.90, sum by (le) (rate(vllm_time_to_first_token_seconds_bucket{{{selector}}}[1m])))",
        "native_ttft_p95_ms": f"1000 * histogram_quantile(0.95, sum by (le) (rate(vllm_time_to_first_token_seconds_bucket{{{selector}}}[1m])))",
        "native_itl_p50_ms": f"1000 * histogram_quantile(0.50, sum by (le) (rate(vllm_inter_token_latency_seconds_bucket{{{selector}}}[1m])))",
        "input_tps": f"sum(rate(vllm_prompt_tokens_total{{{selector}}}[1m]))",
        "output_tps": f"sum(rate(vllm_generation_tokens_total{{{selector}}}[1m]))",
        "kv_cache_utilization": f"avg(vllm_kv_cache_usage_perc{{{selector}}})",
        "queue_depth": f"sum(vllm_num_requests_waiting{{{selector}}})",
        "batch_size": f"sum(vllm_num_requests_running{{{selector}}})",
        "gpu_utilization": f"avg(DCGM_FI_DEV_GPU_UTIL{{{selector}}})",
    }


observability_rows = []
for run in run_windows:
    row = {key: value for key, value in run.items() if key not in {"start", "end"}}
    start = run["start"] - timedelta(seconds=15)
    end = run["end"] + timedelta(seconds=30)
    for metric_name, query in promql_queries(run["endpoint_name"]).items():
        values = metric_values(query_promql_range(query, start, end))
        row[metric_name] = sum(values) / len(values) if values else None
        if metric_name in {"queue_depth", "batch_size", "gpu_utilization", "kv_cache_utilization"}:
            row[f"{metric_name}_max"] = max(values) if values else None
    observability_rows.append(row)

observability_df = pd.DataFrame(observability_rows)
observability_df.sort_values(["target_prefix_tokens", "experiment"])

### Relate cache locality to latency

A useful result should tell one consistent story across layers: lower client TTFT alongside lower native TTFT, stronger KV-cache utilization, and no unacceptable queue or GPU hotspot. A TTFT win paired with sharply higher queue depth indicates that affinity is too sticky for the offered concurrency; lower `ConcurrencyThreshold` and repeat the run.

<!-- PLOT PLACEHOLDER: replace or supplement the generated chart with ./assets/cache-and-queue-depth.png. -->
<!-- ![KV-cache utilization, queue depth, and GPU utilization](./assets/cache-and-queue-depth.png) -->

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for axis, metric, title in zip(
    axes,
    ["kv_cache_utilization", "queue_depth_max", "gpu_utilization"],
    ["Mean KV-cache utilization", "Maximum queue depth", "Mean GPU utilization"],
):
    sns.lineplot(
        data=observability_df,
        x="target_prefix_tokens",
        y=metric,
        hue="experiment",
        marker="o",
        ax=axis,
    )
    axis.set(title=title, xlabel="Shared-prefix tokens", ylabel=metric.replace("_", " "))
plt.tight_layout()
plt.show()

## Validate the quality gate

All validity and task-success columns must be true. Canonical outputs should have one unique value across routing configurations. If a sampled production workload uses nonzero temperature, replace the exact-agreement assertions with your real task evaluator and compare confidence intervals.

In [ ]:
assert quality_df["structured_valid"].all(), "At least one structured output failed validation."
assert quality_df["tool_valid"].all(), "At least one tool call was invalid."
assert quality_df["task_success"].all(), "At least one deterministic task failed."
assert quality_df["structured_output"].nunique() == 1, "Structured output changed across routing configurations."
assert quality_df["tool_output"].nunique() == 1, "Tool-call output changed across routing configurations."
assert quality_df["deterministic_answer"].nunique() == 1, "Temperature-zero answer changed across routing configurations."
quality_df

## Decide when prefix-aware routing is worthwhile

Use the measurements, not the switch alone:

- **Prefer prefix-aware routing** when requests share a long, stable beginning, prefix caching is enabled, TTFT drops at the percentiles your users care about, and KV-cache signals improve without persistent queuing.
- **Lower `ConcurrencyThreshold`** when one affinity group creates queue-depth or tail-latency spikes. This trades some cache locality for load balance.
- **Raise `ConcurrencyThreshold` cautiously** when affinity spills too early and both replicas still have comfortable queue and GPU headroom.
- **Tune `PrefixLength` to the API and payload layout.** It must remain within shared content for affinity, while including enough distinguishing text to distribute independent tenants or prompt families. Remember that native invoke APIs count bytes and the SageMaker OpenAI-compatible API counts message-text characters.
- **Keep RANDOM or least-outstanding-requests** when prefixes are short or unique, requests have highly variable service times, or affinity creates a hotspot larger than the saved prefill work.

Do not generalize AWS's or this notebook's result directly to another model, prompt distribution, instance family, concurrency, or generation length. Repeat the sweep with production-shaped traffic and include endpoint cost per successful request in the final decision.

## Cleanup

The experiment loop deletes each endpoint and endpoint configuration in `finally`. This last cell handles an interrupted current run and removes the shared SageMaker model resource. Run it before closing the notebook.

In [ ]:
from botocore.exceptions import ClientError

if "endpoint_name" in globals():
    try:
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        if status != "Deleting":
            sm.delete_endpoint(EndpointName=endpoint_name)
        sm.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)
    except ClientError as error:
        if error.response["Error"]["Code"] != "ValidationException":
            raise

if "config_name" in globals():
    try:
        sm.delete_endpoint_config(EndpointConfigName=config_name)
    except ClientError as error:
        if error.response["Error"]["Code"] != "ValidationException":
            raise

try:
    sm.delete_model(ModelName=MODEL_NAME)
except ClientError as error:
    if error.response["Error"]["Code"] != "ValidationException":
        raise

print("Cleanup complete.")

## Conclusion and references

We enabled vLLM automatic prefix caching on `Qwen/Qwen3-8B`, kept the model and hardware fixed, and isolated SageMaker routing as the experimental variable. The shared-prefix and concurrency-threshold sweeps reveal both sides of the tradeoff: cache-local prefill can cut TTFT and raise throughput, but affinity should yield before it creates sustained queues. Detailed observability connects the client result to native vLLM cache, latency, token, queue, batch, and GPU behavior, while the quality gate confirms that routing remains an infrastructure optimization rather than an application behavior change.

References:

- [`Qwen/Qwen3-8B`](https://huggingface.co/Qwen/Qwen3-8B)
- [Reduce LLM latency with prefix-aware routing on Amazon SageMaker Inference](https://aws.amazon.com/blogs/machine-learning/reduce-llm-latency-with-prefix-aware-routing-on-amazon-sagemaker-inference/)
- [SageMaker `ProductionVariantRoutingConfig` API](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_ProductionVariantRoutingConfig.html)
- [SageMaker detailed observability for inference](https://docs.aws.amazon.com/sagemaker/latest/dg/monitoring-cloudwatch-detailed-observability.html)
- [SageMaker AI Insights OpenTelemetry metrics](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/SageMaker-AI-Insights-Metrics.html)
- [Query detailed observability metrics with PromQL](https://docs.aws.amazon.com/sagemaker/latest/dg/monitoring-detailed-observability-promql.html)
- [vLLM automatic prefix caching](https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html)
- [Amazon SageMaker Python SDK](https://sagemaker.readthedocs.io/)